# 03 · Structured output, turning text into data

**AI Fundamentals in 3 Hours** · Data Sense

An LLM that returns prose is a nice demo. An LLM that returns a **guaranteed shape** is a
software component you can build on.

In this notebook:

1. Feel the pain of parsing free text
2. See why "please reply in JSON" is not good enough
3. Use `with_structured_output()` to make invalid output impossible
4. Design schemas that actually behave
5. Extract many records from one blob, and batch the whole thing


In [ ]:
# --- run this first, in every notebook ---
import os, json
from pathlib import Path

# read keys out of .env (works from the repo root or from notebooks/)
for candidate in [Path(".env"), Path("../.env")]:
    if candidate.exists():
        for line in candidate.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1-mini"          # provider:model - change this one string to switch providers
model = init_chat_model(MODEL, temperature=0)

import textwrap
def wrap(text, width=88):
    """Print long text wrapped, so answers stay readable on a projector."""
    print(textwrap.fill(str(text), width=width))

assert os.environ.get("OPENAI_API_KEY"), "No API key found - check your .env file"
print("ready |", MODEL)

## 1. The problem

Three real-looking support messages. You want to route them automatically: urgent complaints
to a human, simple questions to a bot, refunds to finance.

First, the naive approach, just ask.

In [ ]:
TICKETS = [
    "Hi, I ordered a table lamp on the 3rd and it still hasn't shipped. This is the second time "
    "this has happened. I want my money back - it was Rs 2,499. Honestly very disappointed.",

    "hey quick q - do you guys deliver to Coimbatore? and how long does it usually take",

    "URGENT!!! My card was charged TWICE for order 91204, Rs 4,150 each time. Please fix this "
    "today, I need the money for rent. Been calling since morning.",
]

wrap(model.invoke(f"Analyse this support ticket:\n\n{TICKETS[0]}").content)

Useful for a human. **Useless for your code.**

Now write the regex that pulls out the amount. Then handle `12,000`, `Rs 12000`, `12k`,
`₹12,000`. Then handle the day it answers in a different format. You will lose a week.

## 2. "Please reply in JSON", better, still not safe

In [ ]:
prompt = f"""Analyse this support ticket and reply with ONLY a JSON object with keys:
sentiment, category, amount_inr, priority.

Ticket: {TICKETS[0]}"""

raw = model.invoke(prompt).content
print("raw response:")
for i in range(0, min(len(raw), 240), 76):
    print(repr(raw[i:i+76]))
print()

try:
    print("parsed:", json.loads(raw))
except json.JSONDecodeError as e:
    print("json.loads FAILED:", e)
    print("A markdown fence, a stray word, and your pipeline is down at 2 a.m.")

It may well have worked just now. Run it a hundred times and it will not work a hundred
times. At 10,000 tickets a day, a 1% parse failure is 100 broken records daily.

## 3. The real answer: `with_structured_output()`

Define the shape in Pydantic, hand it to the model, and the output is **constrained during
generation**. You get back a typed Python object, never a string to parse.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class Ticket(BaseModel):
    """Structured triage record for one customer support ticket."""
    sentiment:   Literal["positive", "neutral", "negative"]
    category:    Literal["refund", "shipping", "payment", "product_question", "other"]
    amount_inr:  Optional[int] = Field(None, description="Money amount mentioned, in rupees. Null if none.")
    priority:    Literal["low", "medium", "high"]
    needs_human: bool = Field(description="True if this needs a human agent rather than a bot.")
    summary:     str  = Field(description="At most 12 words, for the agent's queue view.")


triage = model.with_structured_output(Ticket)       # <-- the whole trick
ticket = triage.invoke(TICKETS[0])

print(type(ticket).__name__)
print()
for field, value in ticket.model_dump().items():
    print(f"  {field:<12}: {value!r}")
print()
print("amount_inr is a real int:", ticket.amount_inr, type(ticket.amount_inr).__name__)

No regex. No `json.loads` in a `try`. No prayer.

And note what `with_structured_output` returned: a **runnable**, just like the model. It has
`.invoke`, `.batch`, `.stream`: the same verbs as everything else in LangChain.

## 4. Run it over all three tickets

Because it is a runnable, `.batch()` runs them **in parallel** for free.

In [ ]:
tickets = triage.batch(TICKETS)      # parallel, not a for-loop

print(f"{'priority':<9}{'category':<17}{'amount':>8}  {'human?':<7}summary")
print("-" * 86)
for t in tickets:
    amt = f"{t.amount_inr:,}" if t.amount_inr else "-"
    summary = t.summary if len(t.summary) <= 46 else t.summary[:45].rsplit(" ", 1)[0] + "…"
    print(f"{t.priority:<9}{t.category:<17}{amt:>8}  {str(t.needs_human):<7}{summary}")

That is a working triage system. You could route on those fields today.

## 5. How to design a schema that behaves

The model **reads your schema:** the class docstring, the field names, and the
descriptions. They are prompt engineering, not documentation.

| Do | Why |
|---|---|
| Use `Literal[...]` enums instead of free strings | Stops it inventing a 14th category next Tuesday |
| Allow `Optional` / `None` | Gives it a way to say "not present" instead of making something up |
| Write a `description` on ambiguous fields | This is where the business rule lives |
| Name fields descriptively (`amount_inr`, not `amt`) | The name is a hint the model uses |
| Keep it flat where you can | Deeply nested schemas degrade quality |

Here is the difference a description makes.

In [ ]:
class Vague(BaseModel):
    urgency: Literal["low", "medium", "high"]

class Precise(BaseModel):
    urgency: Literal["low", "medium", "high"] = Field(
        description=(
            "high = customer has lost money or is blocked right now; "
            "medium = something is wrong but not financially urgent; "
            "low = a question with no problem attached"
        )
    )

msg = TICKETS[1]        # the casual "do you deliver to Coimbatore" one

for schema in (Vague, Precise):
    out = model.with_structured_output(schema).invoke(msg)
    print(f"{schema.__name__:<9} -> {out.urgency}")

print()
print("Without the description you get the model's idea of 'urgent',")
print("which is not your company's idea of urgent.")

## 6. Extraction: many records from one blob

The same mechanism handles lists. This is one of the highest-value things you can do with an
LLM in a real company, turning unstructured text into rows.

In [ ]:
EMAIL = """
Hi team, notes from the Pune vendor meeting:

Sharma Textiles will deliver 400 units of the cotton throw by 28 March, at Rs 640 each.
Kiran Plastics quoted Rs 118 per storage bin, minimum 1000 units, delivery mid-April.
We also spoke to Verma Glass but they can't commit before June so I'd park them.
"""

class Quote(BaseModel):
    vendor: str
    product: str
    unit_price_inr: Optional[int]
    quantity: Optional[int]
    delivery_note: str
    usable: bool = Field(description="False if the vendor cannot commit to a workable timeline.")

class QuoteList(BaseModel):
    """Every vendor quote mentioned in the text."""
    quotes: list[Quote]


out = model.with_structured_output(QuoteList).invoke(
    "Extract every vendor quote. Do not invent values.\n\n" + EMAIL
)

for q in out.quotes:
    price = f"Rs {q.unit_price_inr:,}" if q.unit_price_inr else "no price"
    qty   = f"{q.quantity:,} units" if q.quantity else "qty unknown"
    flag  = "" if q.usable else "   <-- parked"
    print(f"{q.vendor:<17}{q.product:<17}{price:<10}{qty:<13}{q.delivery_note[:18]}{flag}")

Three vendors, correctly separated, with the one that can't commit flagged. From an email.

Try writing that with regex.

## 7. Structured output from an agent

The same `Ticket` schema works on a full agent via `response_format`. The agent can use
tools, think for several steps, and **still** hand you a typed object at the end.

We have not covered tools yet, this is a preview of where notebook 05 leads.

In [ ]:
from langchain.agents import create_agent

triage_agent = create_agent(
    model=model,
    tools=[],                       # tools go here in notebook 05
    system_prompt="You triage customer support tickets for Nimbus Retail.",
    response_format=Ticket,         # <-- typed output from a whole agent
)

result = triage_agent.invoke({"messages": [{"role": "user", "content": TICKETS[2]}]})

print("conversation length :", len(result["messages"]), "messages")
print("structured_response :\n")
for field, value in result["structured_response"].model_dump().items():
    print(f"  {field:<12}: {value!r}")

> **Remember this:** a schema is also how **tool calling** works. Same JSON Schema, same
> mechanism, except instead of constraining the answer, it chooses an action.
>
> That is notebook 05, and it is why this notebook comes first.

## 8. Your turn

1. **Add a field.** Give `Ticket` a `suggested_reply: str` with a description telling it to
   stay under 40 words and never promise a refund. Rerun the triage table.

2. **Break it deliberately.** Change `category` from `Literal[...]` to a plain `str`. Run all
   three tickets a few times and watch the categories drift.

3. **Your own data.** Take any messy text you actually deal with, invoices, WhatsApp
   messages, log lines, job descriptions, and write a schema for it. This is the single most
   immediately useful skill in this workshop.


In [ ]:
# your turn - scratch cell
